# အတန်း ၀၅ - ကိုယ်စားလှယ် RAG


## Setup

ဤ notebook သည် Microsoft Agent Framework ကိုသုံးပြီး Agentic RAG (Retrieval-Augmented Generation) နမူနာကို ဖော်ပြထားသည်။

**လိုအပ်ချက်များ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — သင့်၏ Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — သင့်၏ Azure AI Search API key
- ပတ်ဝန်းကျင် အပြောင်းအလဲများမှတစ်ဆင့် Azure OpenAI deployment ကို ဖွဲ့စည်းထားသည်
- Azure CLI မှ authenticate ပြီး (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG ဆိုတာ ဘာလဲ?

ရိုးရာ RAG မှာ အတည်ပြုထားတဲ့ လုပ်ငန်းစဉ်ကို လိုက်နာတယ်။ အရင်ဆုံး စာရွက်စာတမ်းတွေ ရှာဖွေပြီး၊ ပြန်စာ မေးမြန်းတာကို ထုတ်ပေးတယ်။ **Agentic RAG** ကတော့ ပြင်ပဖြစ်စေတဲ့ အရာတွေကို Agent ကို ကိုယ်ပိုင် ဆုံးဖြတ်ခွင့်ပေးပြီး **အဘယ်အခါ** နဲ့ **ဘယ်လို** အချက်အလက် ရှာဖွေမယ်ဆိုတာကို ရွေးချယ်နိုင်စေတယ်။

Agentic RAG နဲ့ Agent က:
- မေးခွန်းကို ဖြေရှင်းဖို့ မပြုလုပ်ခင် ရှာဖွေဖို့ လိုအပ်ချက် ရှိမရှိကို **ဆုံးဖြတ်** မယ်
- မေးမြန်းဖို့ ဒေတာအရင်းအမြစ် သို့မဟုတ် ကိရိယာကို **ရွေးချယ်** မယ်
- ရှာဖွေပြီးရလာသော ရလဒ်များကို **အကဲဖြတ်** ပြီး ပထမဆုံးကြိုးပမ်းမှု မလုံလောက်ရင် ပိုပြီးရှာဖွေရေးလုပ်ဆောင်မယ်
- စီစဥ်ပြီး ရှာဖွေထားသော အချက်အလက်များကို ပေါင်းစပ်ပြီး ဆက်စပ်တိကျတဲ့ ဖြေ့ချက်တစ်ခုကို **ပေါင်းစပ်** ပြန်ပေးမယ်

ဒီဟာက Agent ကို ပုံမှန် retrieve-then-generate စနစ်ထက် ပိုမိုပြောင်းလဲနိုင်ပြီး တိကျမှန်ကန်မှု ပိုမြင့်စေတယ်။


## ရှာဖွေရေး ကိရိယာ တစ်ခု ဖန်တီးခြင်း

Agentic RAG တွင် ပြင်ပဒေတာအရင်းအမြစ်များကို agent သည် လိုအပ်သလို ခေါ်ယူနိုင်သော **tools** အဖြစ် ထုပ်ပိုးထားပါသည်။ ၎င်းက agent ကို မလိုလားအပ်သောအဆင့်တစ်ခုဖြစ်ပေမယ့် ရယူခြင်းကို ဒါမှမဟုတ် တခြားအရေးယူချက်တစ်ခုအဖြစ် ဆက်လက်ဆောင်ရွက်နိုင်စေသည်။

အောက်တွင် ခရီးသွား အသိပညာအခြေခံကို သတ်မှတ်ပြီး agent သည် ရွှေ့ပြောင်းရာ ဒေသအချက်အလက်များကို ကြည့်ရှုနိုင်ရန် ကိရိယာတစ်ခုအဖြစ် ပြသထားသည်။


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Building the RAG Agent

ယခုတွင် **အမြဲတမ်းဖြေကြားမှုမပြုမီ သတင်းအချက်အလက် ရယူရန် လမ်းညွှန်ထားသော** ကိရိယာတစ်ခုတည်ဆောက်ပါမည်။ ကိရိယာသည် မိမိ၏ သင်ကြားမှုဒေတာပေါ်မတည်၍ အသိပညာအခြေခံခြင်းအတွက် `search_travel_knowledge` ကိရိယာကို အသုံးပြုသည်။


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ပြန်လည်ရှာဖွေရေး အဆင့်ဆင့် — Maker-Checker ပုံစံ

Agentic RAG ၏ အဓိကအားသာချက်မှာ **ပြန်လည်ရှာဖွေရေး အဆင့်ဆင့်** ဖြစ်သည်။ agent သည် သူ၏ ပထမဆုံး ရလဒ်များကို အတည်ပြုရန်၊ ပိုမိုတိကျစေရန် သို့မဟုတ် ကျယ်ပြန့်စေရန် ရှာဖွေမှု အကြိမ်များစွာ ပြုလုပ်နိုင်သည် — "maker-checker" workflowနှင့် ဆင်တူသည်။

1. **Maker အဆင့်**: agent သည် ပထမဆုံး အချက်အလက်များကို ရယူပြီး တုံ့ပြန်ချက် မူကြမ်းရေးဆွဲသည်။
2. **Checker အဆင့်**: agent သည် အသေးစိတ်အချက်အလက်များကို အတည်ပြုရန် သို့မဟုတ် ဖြည့်စွက်ရန် ထပ်မံရှာဖွေသည်။

အောက်တွင်၊ အများကြီး ပစ်မှတ်နေရာများကို နှိုင်းယှဉ်ရန် လိုအပ်သော မေးခွန်းတစ်ခုကို agent ထံမေးထားပြီး အကြိမ်များစွာ ရှာဖွေစေသည်။


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## စာချုပ်

ဤသင်ခန်းစာတွင် Microsoft Agent Framework ကို အသုံးပြု၍ **Agentic RAG** စနစ်တစ်ခုတည်ဆောက်နည်းကို လေ့လာသင်ကြားခဲ့ပါသည်။

- **Agentic RAG** သည် အချက်အလက်ရယူမှုကို agents များအား ကိုယ်တိုင်ဆုံးဖြတ်ခွင့်ပေးကာ ရယူမှုကို ဒိုင်နမစ်ဖြစ်စေပြီး ကြိုတင်သတ်မှတ်ထားခြင်းမရှိစေပါ။
- **ကိရိယာများကို ဒေတာအရင်းအမြစ်များအဖြစ်** အသုံးပြုခြင်း - Azure AI Search ကဲ့သို့ ပြင်ပအသိပညာအခြေခံများကို ကိရိယာများအဖြစ် ထုပ်ပိုးကာ agent အနေဖြင့် ခေါ်ယူနိုင်သည်။
- **အကြိမ်ကြိမ် ရယူခြင်း** - maker-checker ပုံစံက agent ကို ရွာဖွေရေး၊ စစ်ဆေးမှုနှင့် တိုးတတ်ပြုပြင်မှုတို့ကို အဆင့်ဆင့်ပြုလုပ်နိုင်စေပြီး နောက်ဆုံးဖြေကြားချက်ထုတ်ပေးသည်။

ထုတ်လုပ်မှုတွင် in-memory `TRAVEL_KNOWLEDGE_BASE` ကို ပြောင်းလဲ၍ Azure AI Search index တစ်ခုဖြင့် လုပ်ဆောင်ကာ သွားလာရေးစာရွက်စာတမ်းများကြီးစွာ ရယူနိုင်ရန် အသုံးပြုလိမ့်မည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အနှုတ်ချုပ်**  
ဤစာတမ်းကို AI ဘာသာပြန်မှုဝန်ဆောင်မှုဖြစ်သော [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် မှန်ကန်မှုအတွက် ကြိုးစားဆောင်ရွက်သော်လည်း၊ အလိုအလျောက် ပြန်ဆိုမှုများတွင် အမှားများ သို့မဟုတ် မှန်ကန်မှုလျော့နည်းမှုများ ရှိနိုင်ကြောင်း သတိပြုပါ။ မူရင်းစာတမ်းကို မိသားစုဘာသာဖြင့်သာ ယုံကြည်စရာ အခြေခံအရင်းအမြစ်အဖြစ် ယူဆသင့်ပါသည်။ အရေးပါတ်သော သတင်းအချက်အလက်များအတွက် ပညာရှင်လူ့ဘာသာပြန်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်မှုကို အသုံးပြုသည့်အတွက် ဖြစ်ပေါ်နိုင်သည့် နားမလည်မှုများ သို့မဟုတ် မှားယွင်းနားဖြတ်မှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
